# Notebook Overview

This notebook prepares the **GraSCCo PII** dataset for later de-identification experiments and evaluation.

**Main outputs**
- `data_regex/processed/grascco_canonical.jsonl` (one document per line)

Run the cells from top to bottom. The first cell is only a quick check to see whether a GPU is available.


In [ ]:

import torch

print("CUDA verfügbar:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Keine GPU gefunden")

# GraSCCo Data Preparation Notebook

This notebook prepares the **GraSCCo_PII_V2** dataset for evaluation. It performs the following tasks:

1. **Load the INCEpTION JSON files** from the GraSCCo zip.
2. **Extract the raw clinical text** for each document.
3. **Extract the gold PHI spans** annotated in the dataset (begin, end, kind) and optionally map them to broader categories.
4. **Validate** the spans to ensure offsets are within the text and non-empty.
5. **Write a canonical JSONL file** where each line represents a document with its text and gold spans.

*Note:* This notebook **does not perform anonymization or de-identification**. The gold spans are only used for evaluation and are not fed to any model. A separate pipeline (e.g. Regex + LLM) should process the text and then be evaluated against these gold spans.


In [ ]:

# Here we define the basic project paths and make the run reproducible using a fixed random seed.

from pathlib import Path
import random

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

DATA_RAW = Path("data_regex/raw")
DATA_PROC = Path("data_regex/processed")
DATA_PROC.mkdir(parents=True, exist_ok=True)

ZIP_JSON = DATA_RAW / "grascco_pii_2_json.zip"

assert ZIP_JSON.exists(), f"Missing: {ZIP_JSON}. Files found: {[p.name for p in DATA_RAW.iterdir()]}"
print("Found dataset zip:", ZIP_JSON)


In [ ]:

# Small sanity check to confirm that the raw data folder exists and contains the dataset archive.

print("DATA_RAW exists:", DATA_RAW.exists())
print("Files in data_regex/raw:", list(DATA_RAW.iterdir()))


## Inspecting the ZIP 
You can inspect how many JSON files are inside the zip. Each JSON file represents one document.


In [ ]:

# Optional inspection step: list JSON files inside the dataset ZIP to confirm its structure.

with zipfile.ZipFile(ZIP_JSON, "r") as zf:
    json_files = [n for n in zf.namelist() if n.lower().endswith(".json")]
    print(f"Number of JSON files in the zip: {len(json_files)}")
    print("First 5 files:", json_files[:5])


## Parsing the INCEpTION JSON structure

The INCEpTION export uses a UIMA CAS structure encoded in JSON. Each file contains a `%FEATURE_STRUCTURES` list holding all annotation objects. We need to extract:

- The **text**: It's stored in the structure with `%TYPE` equal to `"uima.cas.Sofa"` under the key `sofaString`.
- The **gold PHI spans**: Each span is an object where `%TYPE` is `"webanno.custom.PHI"`. These objects have the keys `begin`, `end`, and `kind`, which correspond to the start offset, end offset, and PHI type, respectively.
- The **document ID or title**: There is usually a structure of type `DocumentMetaData` that stores `documentId` or `documentTitle`. We use it to identify each document.

We ignore other annotation types (e.g. Token, Sentence, POS), since they are not required for evaluation.


In [ ]:
# This cell defines helper functions to parse the INCEpTION (UIMA CAS) JSON format.
# The result is a simplified per-document structure with the raw text and gold PHI spans.


def parse_inception_json(obj, fallback_doc_id=""):
    # Parse a single INCEpTION JSON object into a simple dictionary.
    # Extracts text from uima.cas.Sofa, PHI spans from webanno.custom.PHI, and document ID/title.

    # Find text
    text = None
    sofa_id = None
    for fs in obj.get("%FEATURE_STRUCTURES", []):
        if fs.get("%TYPE") == "uima.cas.Sofa":
            text = fs.get("sofaString")
            sofa_id = fs.get("%ID")
            break
    if text is None:
        return None

    # Find document ID or title
    doc_id = None
    doc_title = None
    for fs in obj.get("%FEATURE_STRUCTURES", []):
        if fs.get("%TYPE", "").endswith("DocumentMetaData"):
            doc_id = fs.get("documentId")
            doc_title = fs.get("documentTitle")
            break
    doc_key = (doc_title or doc_id or fallback_doc_id)
    doc_key = str(doc_key)

    # Collect PHI spans
    spans = []
    for fs in obj.get("%FEATURE_STRUCTURES", []):
        if fs.get("%TYPE") == "webanno.custom.PHI":
            b = fs.get("begin")
            e = fs.get("end")
            kind = fs.get("kind")
            if b is None or e is None or kind is None:
                continue
            # If @sofa is present, ensure it matches the main sofa
            if sofa_id is not None and fs.get("@sofa") is not None and fs.get("@sofa") != sofa_id:
                continue
            spans.append({"start": int(b), "end": int(e), "label_raw": str(kind)})
    return {"doc_id": doc_key, "text": text, "spans_raw": spans}


def validate_spans(record):
    # Validate spans: keep only those within text bounds and non-empty
    text = record["text"]
    clean = []
    for s in record["spans_raw"]:
        b, e = s["start"], s["end"]
        if not (0 <= b < e <= len(text)):
            continue
        if not text[b:e].strip():
            continue
        clean.append(s)
    record["spans_raw"] = clean
    return record


## Optional label mapping

The original `kind` values in the dataset can be kept as-is or mapped to broader categories. For example, you might map all variants of `NAME` (patient, doctor) to a single `NAME` label, or keep them separate for fine-grained analysis. Adjust the `LABEL_MAP` below as needed.


In [ ]:
# Optional: define a label mapping to collapse very fine-grained PHI types into broader categories.
# If LABEL_MAP is empty, labels are kept unchanged.

# Define your label mapping here if desired.
# If a raw label is not present in the mapping, it will be kept unchanged.
LABEL_MAP = {
    # Example mappings:
    # 'NAME_PATIENT': 'NAME',
    # 'NAME_DOCTOR': 'NAME',
    # 'DATE_BIRTH': 'DATE',
    # 'LOCATION_CITY': 'LOCATION',
}

def map_label(raw):
    return LABEL_MAP.get(raw, raw)


## Parse all documents and build canonical list

This step loops over all JSON files, parses each into a canonical structure, validates spans, applies the label mapping, and appends to a list.


In [ ]:
# Main extraction step:
# - read each JSON document from the ZIP
# - parse text + spans
# - validate offsets
# - apply label mapping
# The result is stored in the list `canonical`.

canonical = []
with zipfile.ZipFile(ZIP_JSON, "r") as zf:
    json_files = [n for n in zf.namelist() if n.lower().endswith(".json")]
    for name in json_files:
        obj = json.loads(zf.read(name).decode("utf-8"))
        rec = parse_inception_json(obj, fallback_doc_id=name)
        if rec is None:
            continue
        rec = validate_spans(rec)
        # Apply label mapping
        spans = []
        for s in rec["spans_raw"]:
            spans.append({
                "start": s["start"],
                "end": s["end"],
                "label": map_label(s["label_raw"]),
                "label_raw": s["label_raw"],
            })
        canonical.append({
            "doc_id": rec["doc_id"],
            "text": rec["text"],
            "spans": [{"start": s["start"], "end": s["end"], "label": s["label"]} for s in spans],
            "spans_raw": [{"start": s["start"], "end": s["end"], "label_raw": s["label_raw"]} for s in spans],
        })

print("Number of documents parsed:", len(canonical))
print("Example document ID:", canonical[0]["doc_id"] if canonical else "n/a")
print("Number of spans in example:", len(canonical[0]["spans"]) if canonical else 0)


## Save the canonical dataset

We write each document as a single JSON object per line in a `.jsonl` file. This file can then be used for evaluation or further processing. The raw labels are kept in `spans_raw` for traceability.


In [ ]:
# Save the canonical dataset to disk.
# JSONL is recommended for larger datasets because it can be streamed line by line.

OUT_CANON = DATA_PROC / "grascco_canonical.jsonl"
with OUT_CANON.open("w", encoding="utf-8") as f:
    for rec in canonical:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print("Written canonical JSONL to:", OUT_CANON.resolve())

OUT_JSON = DATA_PROC / "grascco_canonical.json"
with OUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(canonical, f, ensure_ascii=False, indent=2)
print("Written canonical JSON to:", OUT_JSON.resolve())


## Quick dataset statistics

We compute simple statistics such as the average document length, average number of spans per document, and the distribution of labels. This is useful to understand the dataset composition.


In [ ]:
# Quick descriptive statistics to understand dataset size and label distribution.

label_counts = Counter()
lengths = []
spans_per_doc = []
for rec in canonical:
    lengths.append(len(rec["text"]))
    spans_per_doc.append(len(rec["spans"]))
    for s in rec["spans"]:
        label_counts[s["label"]] += 1

print("Documents:", len(canonical))
print("Average characters per document:", sum(lengths)/len(lengths) if lengths else 0)
print("Average spans per document:", sum(spans_per_doc)/len(spans_per_doc) if spans_per_doc else 0)
print("Top 10 labels (after mapping):")
for lab, cnt in label_counts.most_common(10):
    print(f"{lab:20s} {cnt}")
